# Matcher -> OpenVINO IR export (reusable, on disk)

Exports a PyTorch `Matcher` to OpenVINO IR files you can reload later, then runs inference on CPU.

Two reusable sub-models are produced (reference features are inputs, not baked in):

* `encoder.xml` - image -> patch embeddings (used for reference and target)
* `head.xml` - (target_image, target_embeddings, ref_features...) -> masks/scores/labels
* `metadata.json` - input size / patch size used by `MatcherOpenVINO`

For a zero-boilerplate temp-dir flow see `matcher_openvino_example.ipynb` (`MatcherOpenVINO.from_torch`).

In [ ]:
from pathlib import Path

from instantlearn.models import Matcher, MatcherOpenVINO
from instantlearn.models.torch_base import ExportConfig
from instantlearn.scripts.matcher import export_matcher
from instantlearn.data.base.sample import Category, Sample
from instantlearn.utils.constants import CompressionMode

EXPORT_DIR = Path("./matcher-openvino")
REF_IMAGE = "assets/coco/000000286874.jpg"
REF_MASK = "assets/coco/000000286874_mask.png"
TARGET_IMAGE = "assets/coco/000000390341.jpg"

## 1. Export the IR to disk (INT8 by default)

The model does not need to be fitted before export.

In [ ]:
matcher = Matcher(device="cpu")
paths = export_matcher(
    matcher,
    output_dir=EXPORT_DIR,
    config=ExportConfig(compression=CompressionMode.INT8_SYM),
)
for name, path in paths.items():
    print(f"{name}: {path}")

## 2. Reload the IR and run inference

`MatcherOpenVINO(model_dir=...)` loads the exported files. `fit()` encodes the reference through `encoder.xml`; `predict()` returns `Prediction` objects.

In [ ]:
ov_model = MatcherOpenVINO(model_dir=EXPORT_DIR, device="CPU")
ov_model.fit(Sample(image_path=REF_IMAGE, mask_paths=[REF_MASK], categories=[Category(0, "elephant")]))
pred = ov_model.predict(Sample(image_path=TARGET_IMAGE))[0]
print("masks :", pred.masks.shape)
print("scores:", pred.scores)
print("labels:", pred.label_names)

## 3. Visualize

In [ ]:
import cv2
import matplotlib.pyplot as plt
from instantlearn.visualizer import render_predictions, setup_colors

target_rgb = cv2.cvtColor(cv2.imread(TARGET_IMAGE), cv2.COLOR_BGR2RGB)
vis = render_predictions(target_rgb, pred, setup_colors({0: "elephant"}))
plt.figure(figsize=(10, 8))
plt.imshow(vis)
plt.axis("off")
plt.show()